# CC Media Studio · Video Preview
选择 GPU T4 ×2，开启 Internet，然后运行全部单元格。 / Select GPU T4 ×2, enable Internet, and run all cells.
首次自动下载模型；后续请求复用当前模型。MP4 保存到 `/kaggle/working/cc-media-video.mp4`。 / First use downloads weights; subsequent requests reuse the active model.
结束后在 Kaggle 停止 GPU 会话。 / Stop the Kaggle GPU session when finished.

In [ ]:
import sys,subprocess,site,json
from pathlib import Path
environment=Path('/kaggle/temp/cc-video-environment')
if not environment.exists():
    subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(environment)],check=True)
subprocess.run([str(environment/'bin/python'),'-m','pip','install','-q',
    'diffusers==0.36.0','transformers==4.57.6','accelerate==1.12.0',
    'bitsandbytes==0.50.2','sentencepiece','imageio-ffmpeg'],check=True)
packages=environment/'lib'/('python'+str(sys.version_info.major)+'.'+str(sys.version_info.minor))/'site-packages'
sys.path.insert(0,str(packages))
modules=Path('/kaggle/temp/cc-video-modules');modules.mkdir(exist_ok=True)
for name,source in {'asset_manager.py': '"""Download public HF weights with resumable transfers and real byte progress."""\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nfrom pathlib import Path, PurePosixPath\nfrom urllib.parse import quote\nimport requests\n\nBASE_REPO = \'Qwen/Qwen-Image-2.1\'\nGGUF_REPO = \'azukivc/Qwen-Image-2.1-GGUF\'\nBASE_REVISION = \'d26bb61231c349cf6b7896fa83353113880e1ba3\'\nGGUF_REVISION = \'4e43fd824763d3d24233630c343462592f68ca49\'\n\n\ndef emit_progress(phase, message, done=None, total=None, filename=None):\n    event = {\'phase\': phase, \'message\': message}\n    if done is not None: event[\'downloaded_bytes\'] = done\n    if total: event.update(total_bytes=total, percent=round(min(100, 100*done/total), 1))\n    if filename: event[\'file\'] = filename\n    callback = globals().get(\'_QWEN_PROGRESS_CALLBACK\')\n    if callback: callback(event)\n    print(\'COMFY_PROGRESS:\' + json.dumps(event, ensure_ascii=False), flush=True)\n\n\ndef repository_manifest(repo, session=None, revision=None):\n    http = session or requests.Session()\n    endpoint = \'https://huggingface.co/api/models/\' + repo\n    if revision: endpoint += \'/revision/\' + revision\n    response = http.get(endpoint, params={\'blobs\':\'true\'}, timeout=(20,60))\n    response.raise_for_status()\n    data = response.json()\n    if data.get(\'private\') or data.get(\'gated\'):\n        raise RuntimeError(\'默认权重必须能够公开下载，当前来源需要额外授权。\')\n    return data\n\n\ndef safe_relative(name):\n    path = PurePosixPath(name)\n    if path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name:\n        raise ValueError(\'Invalid model file path\')\n    return path\n\n\ndef download_file(http, url, destination, expected_size, expected_sha=None, progress=None, verify_progress=None):\n    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_name(destination.name + \'.partial\')\n    receipt = destination.with_name(destination.name + \'.verified.json\')\n    if destination.is_file() and destination.stat().st_size == expected_size:\n        try:\n            stamp = json.loads(receipt.read_text())\n            if stamp == {\'size\':expected_size, \'sha256\':expected_sha}:\n                if progress: progress(expected_size)\n                return\n        except (OSError,ValueError): pass\n        # Existing files without a verification receipt must still be checked.\n        partial.unlink(missing_ok=True); destination.replace(partial)\n    for attempt in range(3):\n        try:\n            offset = partial.stat().st_size if partial.exists() else 0\n            if expected_size and offset > expected_size:\n                partial.unlink(); offset = 0\n            if offset != expected_size or not partial.exists():\n                headers = {\'Range\':f\'bytes={offset}-\'} if offset else {}\n                with http.get(url, headers=headers, stream=True, timeout=(20,60)) as response:\n                    response.raise_for_status()\n                    if offset and response.status_code == 206:\n                        if not response.headers.get(\'Content-Range\',\'\').startswith(f\'bytes {offset}-\'):\n                            raise ValueError(\'Invalid download resume range\')\n                    else:\n                        offset = 0\n                    with partial.open(\'ab\' if offset else \'wb\') as output:\n                        for chunk in response.iter_content(chunk_size=4*1024*1024):\n                            if chunk:\n                                output.write(chunk); offset += len(chunk)\n                                if progress: progress(offset)\n            if expected_size and partial.stat().st_size != expected_size:\n                raise OSError(\'Incomplete model download\')\n            if expected_sha:\n                digest = hashlib.sha256()\n                with partial.open(\'rb\') as content:\n                    verified = 0\n                    for chunk in iter(lambda:content.read(16*1024*1024), b\'\'):\n                        digest.update(chunk); verified += len(chunk)\n                        if verify_progress: verify_progress(verified)\n                if digest.hexdigest() != expected_sha:\n                    partial.unlink(); raise ValueError(\'Model checksum mismatch\')\n            partial.replace(destination)\n            receipt.write_text(json.dumps({\'size\':expected_size,\'sha256\':expected_sha}))\n            if progress: progress(destination.stat().st_size)\n            return\n        except (requests.RequestException, OSError, ValueError):\n            if attempt == 2: raise\n            time.sleep(2*(attempt+1))\n\n\ndef prepare_hf_assets(folder, http=None):\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    http = http or requests.Session()\n    emit_progress(\'manifest\', \'正在从 Hugging Face 获取模型文件清单。\')\n    base = repository_manifest(BASE_REPO, http, BASE_REVISION)\n    quant = repository_manifest(GGUF_REPO, http, GGUF_REVISION)\n    records = []\n    for item in base[\'siblings\']:\n        name = item[\'rfilename\']\n        if name in {\'model_index.json\',\'LICENSE\',\'transformer/config.json\'} or any(name.startswith(p+\'/\') for p in [\'processor\',\'scheduler\',\'vae\',\'text_encoder\']):\n            if name.endswith(\'.json\') or name.endswith(\'.safetensors\') or name.endswith(\'.txt\') or name.endswith(\'.model\') or name.endswith(\'.jinja\') or name.endswith(\'.tiktoken\') or name == \'LICENSE\':\n                records.append((BASE_REPO,base[\'sha\'],item,name))\n    candidates = [f for f in quant[\'siblings\'] if f[\'rfilename\'].lower().endswith(\'q4_k_m.gguf\')]\n    if len(candidates) != 1: raise RuntimeError(\'兼容的 Q4_K_M 权重清单不明确。\')\n    records.append((GGUF_REPO,quant[\'sha\'],candidates[0],\'transformer-q4.gguf\'))\n    if not any(name.startswith(\'text_encoder/\') and name.endswith(\'.safetensors\') for _,_,_,name in records):\n        raise RuntimeError(\'模型清单缺少文本编码器权重。\')\n    total = sum(int(item.get(\'size\') or item.get(\'lfs\',{}).get(\'size\') or 0) for _,_,item,_ in records)\n    cached=True\n    for _,_,item,name in records:\n        safe_relative(name)\n        size=int(item.get(\'size\') or item.get(\'lfs\',{}).get(\'size\') or 0)\n        target=folder/name\n        try:\n            receipt=json.loads(target.with_name(target.name+\'.verified.json\').read_text())\n            if not target.is_file() or target.stat().st_size!=size or receipt!={\'size\':size,\'sha256\':item.get(\'lfs\',{}).get(\'sha256\')}:\n                cached=False;break\n        except (OSError,ValueError):cached=False;break\n    if cached:\n        emit_progress(\'cached\',\'已找到完整模型缓存，无需重新下载。\',total,total or None)\n        return folder\n    done = 0; last = [0.0]\n    for repo,revision,item,name in records:\n        safe_relative(name)\n        size = int(item.get(\'size\') or item.get(\'lfs\',{}).get(\'size\') or 0)\n        sha = item.get(\'lfs\',{}).get(\'sha256\')\n        start = done\n        def progress(value, name=name, start=start):\n            now=time.monotonic()\n            if now-last[0]>=.8 or value==size:\n                emit_progress(\'download\',\'首次准备：正在下载模型权重。\',start+value,total or None,name)\n                last[0]=now\n        url=\'https://huggingface.co/\'+repo+\'/resolve/\'+revision+\'/\'+quote(item[\'rfilename\'],safe=\'/\')\n        def verify_progress(value, name=name, size=size):\n            now=time.monotonic()\n            if now-last[0]>=.8 or value==size:\n                emit_progress(\'verify\',\'正在校验下载的权重。\',value,size,name);last[0]=now\n        download_file(http,url,folder/name,size,sha,progress,verify_progress)\n        done += size\n    (folder/\'asset-sources.json\').write_text(json.dumps({\'base_repo\':BASE_REPO,\'base_revision\':base[\'sha\'],\'gguf_repo\':GGUF_REPO,\'gguf_revision\':quant[\'sha\'],\'download_bytes\':total},indent=2))\n    emit_progress(\'download\',\'模型权重下载完成。\',done,total or None)\n    return folder\n\n\ndef copy_assets(source, target):\n    source,target=Path(source),Path(target)\n    files=[p for p in source.rglob(\'*\') if p.is_file()]\n    total=sum(p.stat().st_size for p in files); done=0; last=0\n    for path in files:\n        destination=target/path.relative_to(source); destination.parent.mkdir(parents=True,exist_ok=True)\n        with path.open(\'rb\') as src, destination.open(\'wb\') as dst:\n            for chunk in iter(lambda:src.read(8*1024*1024),b\'\'):\n                dst.write(chunk);done+=len(chunk)\n                now=time.monotonic()\n                if now-last>=.8:\n                    emit_progress(\'copy\',\'正在准备已缓存的模型文件。\',done,total,path.name);last=now\n    emit_progress(\'copy\',\'模型文件准备完成。\',done,total)\n', 'video_precision.py': '"""Explicit device moves for INT8 video encoders without changing image helpers."""\n\n\ndef move_video_encoder(model,device):\n    import torch\n    from bitsandbytes.nn import Linear8bitLt\n    target=torch.device(device)\n    if all(p.device==target for p in model.parameters()):return model\n    layers=[]\n    for module in model.modules():\n        if not isinstance(module,Linear8bitLt):continue\n        scale=module.state.SCB if module.state.SCB is not None else module.weight.SCB\n        if scale is None:raise RuntimeError(\'Missing INT8 video scale\')\n        layers.append((module,scale))\n    for module,_ in layers:\n        module.state.CB=None;module.state.SCB=None;module.weight.CB=None;module.weight.SCB=None\n    try:\n        # Transformers forbids .to() on some 8-bit model classes even though\n        # torch/bitsandbytes support moving their quantized parameters.\n        return torch.nn.Module.to(model,target)\n    finally:\n        for module,scale in layers:\n            module.state.CB=module.weight.data;module.state.SCB=scale.to(module.weight.device)\n            module.weight.CB=None;module.weight.SCB=None\n', 'video_registry.py': '"""Video-only catalog; ordinary image clients do not import GPU packages."""\nVIDEO_MODELS = {\n    \'ltx-video-2b-distilled\': {\n        \'id\':\'ltx-video-2b-distilled\',\'name\':\'LTX-Video 2B · Efficient\',\'tier\':\'efficient\',\n        \'family\':\'ltx\',\'default_steps\':8,\'fixed_steps\':8,\'width\':768,\'height\':448,\n        \'frames\':33,\'fps\':16,\'guidance\':1.0,\'cache_slug\':\'cc-media-ltx-2b-distilled\',\n        \'repo\':\'Lightricks/LTX-Video\',\'revision\':\'8984fa25007f376c1a299016d0957a37a2f797bb\',\n        \'checkpoint\':\'ltxv-2b-0.9.6-distilled-04-25.safetensors\'},\n    \'wan21-t2v-1.3b\': {\n        \'id\':\'wan21-t2v-1.3b\',\'name\':\'Wan2.1 1.3B · Balanced\',\'tier\':\'balanced\',\n        \'family\':\'wan\',\'default_steps\':30,\'fixed_steps\':None,\'width\':832,\'height\':480,\n        \'frames\':33,\'fps\':16,\'guidance\':5.0,\'cache_slug\':\'cc-media-wan21-1-3b\',\n        \'repo\':\'Wan-AI/Wan2.1-T2V-1.3B-Diffusers\',\'revision\':\'0fad780a534b6463e45facd96134c9f345acfa5b\'},\n    \'wan22-ti2v-5b\': {\n        \'id\':\'wan22-ti2v-5b\',\'name\':\'Wan2.2 5B · Quality\',\'tier\':\'quality\',\n        \'family\':\'wan22\',\'default_steps\':30,\'fixed_steps\':None,\'width\':768,\'height\':448,\n        \'frames\':33,\'fps\':24,\'guidance\':5.0,\'cache_slug\':\'cc-media-wan22-5b\',\n        \'repo\':\'Wan-AI/Wan2.2-TI2V-5B-Diffusers\',\'revision\':\'b8fff7315c768468a5333511427288870b2e9635\'},\n}\n\n\ndef video_profile(model_id=\'wan21-t2v-1.3b\'):\n    if not isinstance(model_id,str) or model_id not in VIDEO_MODELS:\n        raise ValueError(\'未知视频模型 / Unknown video model.\')\n    return dict(VIDEO_MODELS[model_id])\n\n\ndef video_catalog():\n    return [{k:v for k,v in p.items() if k not in (\'repo\',\'revision\',\'checkpoint\')}\n            for p in VIDEO_MODELS.values()]\n\n\ndef validate_video(prompt,seed=42,steps=None,model_id=\'wan21-t2v-1.3b\',frames=None):\n    p=video_profile(model_id)\n    if not isinstance(prompt,str) or not 1<=len(prompt.strip())<=4000:\n        raise ValueError(\'提示词需要 1–4000 个字符。\')\n    if type(seed) is not int or not 0<=seed<=2147483647:raise ValueError(\'Invalid seed\')\n    steps=p[\'default_steps\'] if steps is None else steps\n    frames=p[\'frames\'] if frames is None else frames\n    if type(steps) is not int or not 1<=steps<=50:raise ValueError(\'steps must be 1–50\')\n    if p[\'fixed_steps\'] and steps!=p[\'fixed_steps\']:raise ValueError(\'LTX distilled requires 8 steps\')\n    if type(frames) is not int or not 9<=frames<=81 or (frames-1)%8:\n        raise ValueError(\'frames must be 9, 17, …, 81\')\n    return {\'prompt\':prompt,\'seed\':seed,\'steps\':steps,\'model_id\':p[\'id\'],\'frames\':frames}\n', 'video_assets.py': '"""Pinned, resumable video downloads using the existing checked transfer helper."""\nimport json\nimport time\nfrom pathlib import Path\nfrom urllib.parse import quote\n\n\ndef video_asset_records(profile,manifest):\n    if manifest.get(\'sha\')!=profile[\'revision\']:raise RuntimeError(\'Video revision mismatch\')\n    records=[]\n    for item in manifest[\'siblings\']:\n        name=item[\'rfilename\']\n        from asset_manager import safe_relative\n        safe_relative(name)\n        parts=name.split(\'/\')\n        metadata=name in (\'model_index.json\',\'README.md\') or (\'license\' in name.lower() and len(parts)==1)\n        component=parts[0] in (\'text_encoder\',\'tokenizer\',\'transformer\',\'scheduler\',\'vae\')\n        keep=metadata or (component and name.endswith((\'.json\',\'.txt\',\'.model\',\'.safetensors\')))\n        if profile[\'family\']==\'ltx\':\n            keep=metadata or name==profile[\'checkpoint\'] or (component and\n                (not name.endswith(\'.safetensors\') or parts[0]==\'text_encoder\'))\n        if keep:records.append(item)\n    if not any(i[\'rfilename\'].endswith(\'.safetensors\') for i in records):raise RuntimeError(\'Missing video weights\')\n    return records\n\n\ndef prepare_video_assets(profile,folder,progress=None):\n    import requests\n    from asset_manager import repository_manifest,download_file\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    notify=progress or (lambda event:None)\n    notify({\'phase\':\'manifest\'})\n    with requests.Session() as http:\n        records=video_asset_records(profile,repository_manifest(profile[\'repo\'],http,profile[\'revision\']))\n        sizes=[int(i.get(\'size\') or i.get(\'lfs\',{}).get(\'size\') or 0) for i in records]\n        if any(size<=0 for size in sizes):raise RuntimeError(\'Missing video file size\')\n        total=sum(sizes);done=0;last=[0.0]\n        for item,size in zip(records,sizes):\n            name=item[\'rfilename\'];start=done\n            def report(value,start=start,name=name,size=size):\n                now=time.monotonic()\n                if now-last[0]>=1 or value==size:\n                    notify(dict(phase=\'download\',file=name,downloaded_bytes=start+value,\n                        total_bytes=total,percent=round(100*(start+value)/total,1)));last[0]=now\n            url=\'https://huggingface.co/\'+profile[\'repo\']+\'/resolve/\'+profile[\'revision\']+\'/\'+quote(name,safe=\'/\')\n            download_file(http,url,folder/name,size,item.get(\'lfs\',{}).get(\'sha256\'),report)\n            done+=size\n    (folder/\'asset-sources.json\').write_text(json.dumps({\'studio_model_id\':profile[\'id\'],\n        \'model_sources\':[[profile[\'repo\'],profile[\'revision\']]],\'download_bytes\':total}),encoding=\'utf-8\')\n    return folder\n', 'video_parallel.py': '"""Transactional two-GPU CFG for Wan\'s separate conditional/unconditional calls."""\nfrom concurrent.futures import ThreadPoolExecutor\nfrom contextlib import nullcontext\nfrom functools import wraps\n\n\nclass WanParallelCFG:\n    def __init__(self,primary,secondary,negative,torch_module=None):\n        if torch_module is None:\n            import torch as torch_module\n        self.torch=torch_module;self.primary=primary;self.secondary=secondary\n        self.negative=negative.to(\'cuda:1\');self.pending=None;self.calls=0\n\n    def __enter__(self):\n        self.original=self.primary.forward\n        self.pool=ThreadPoolExecutor(max_workers=1,thread_name_prefix=\'cc-video-cfg\')\n        @wraps(self.original)\n        def forward(*args,**kwargs):\n            if args or kwargs.get(\'return_dict\') is not False:\n                raise ValueError(\'Unsupported Wan CFG forward signature\')\n            if self.pending is not None:\n                pending,self.pending=self.pending,None\n                output,event=pending.result();event.synchronize();self.calls+=1\n                return tuple(x.to(\'cuda:0\') if hasattr(x,\'to\') else x for x in output)\n            copy={k:self._copy(v) for k,v in kwargs.items()}\n            copy[\'encoder_hidden_states\']=self.negative\n            # Finish peer copies before launching either branch on another thread.\n            self.torch.cuda.synchronize(1)\n            self.pending=self.pool.submit(self._unconditional,copy)\n            return self.original(**kwargs)\n        self.primary.forward=forward\n        return self\n\n    def _copy(self,value):\n        if self.torch.is_tensor(value):return value.to(\'cuda:1\')\n        if isinstance(value,dict):return {k:self._copy(v) for k,v in value.items()}\n        if isinstance(value,(tuple,list)):return type(value)(self._copy(v) for v in value)\n        return value\n\n    def _unconditional(self,kwargs):\n        with self.torch.inference_mode(),self.torch.cuda.device(1):\n            context=getattr(self.secondary,\'cache_context\',None)\n            with context(\'uncond\') if context else nullcontext():output=self.secondary(**kwargs)\n            event=self.torch.cuda.Event();event.record(self.torch.cuda.current_stream(1))\n            return output,event\n\n    def __exit__(self,kind,*rest):\n        self.primary.forward=self.original\n        self.pool.shutdown(wait=True,cancel_futures=True)\n        unpaired=self.pending is not None;self.pending=None\n        if kind is None and (unpaired or self.calls==0):raise RuntimeError(\'Unpaired Wan CFG calls\')\n', 'video_runtime.py': '"""GPU video runtime. Imported only on the remote Kaggle kernel."""\nimport gc\nimport json\nimport time\nfrom pathlib import Path\n\n\nclass VideoRuntime:\n    def __init__(self,model_id=\'wan21-t2v-1.3b\',acceleration=\'auto\',\n                 root=\'/kaggle/temp/cc-media-video\',progress=None):\n        from video_registry import video_profile\n        if acceleration not in (\'auto\',\'split\'):raise ValueError(\'Invalid acceleration\')\n        self.profile=video_profile(model_id);self.acceleration=acceleration\n        self.folder=Path(root)/model_id;self.progress=progress or (lambda e:None)\n        self.pipe=None;self.encoder=None;self.replica=None;self.calls=0;self.disabled=False\n\n    def emit(self,phase,**extra):self.progress(dict(phase=phase,**extra))\n\n    def prepare(self):\n        import torch\n        from transformers import BitsAndBytesConfig,T5EncoderModel,UMT5EncoderModel\n        from diffusers import (LTXPipeline,WanPipeline,\n                               WanTransformer3DModel,AutoencoderKLWan)\n        from video_assets import prepare_video_assets\n        if torch.cuda.device_count()<2:raise RuntimeError(\'Select GPU T4 x2\')\n        start=time.perf_counter();p=self.profile\n        prepare_video_assets(p,self.folder,self.progress)\n        self.emit(\'loading\')\n        encoder_type=T5EncoderModel if p[\'family\']==\'ltx\' else UMT5EncoderModel\n        self.encoder=encoder_type.from_pretrained(self.folder/\'text_encoder\',\n            torch_dtype=torch.float16,device_map={\'\':1},low_cpu_mem_usage=True,\n            quantization_config=BitsAndBytesConfig(load_in_8bit=True),local_files_only=True)\n        if p[\'family\']==\'ltx\':\n            self.pipe=LTXPipeline.from_single_file(str(self.folder/p[\'checkpoint\']),\n                config=str(self.folder),text_encoder=self.encoder,torch_dtype=torch.float16,\n                local_files_only=True)\n            self.pipe.transformer.to(\'cuda:0\');self.pipe.vae.to(\'cuda:0\')\n        else:\n            self.pipe_class=WanPipeline\n            self.transformer_type=WanTransformer3DModel\n            self.pipe=self.pipe_class.from_pretrained(self.folder,text_encoder=self.encoder,\n                transformer=self.load_transformer(0),\n                vae=AutoencoderKLWan.from_pretrained(self.folder/\'vae\',torch_dtype=torch.float32,\n                    local_files_only=True).to(\'cuda:0\'),torch_dtype=torch.float16,local_files_only=True)\n        if hasattr(self.pipe.vae,\'enable_tiling\'):self.pipe.vae.enable_tiling()\n        if hasattr(self.pipe.vae,\'enable_slicing\'):self.pipe.vae.enable_slicing()\n        self.load_seconds=time.perf_counter()-start\n        return self\n\n    def load_transformer(self,device):\n        import torch\n        from diffusers import BitsAndBytesConfig\n        kwargs={}\n        if self.profile[\'family\']==\'wan22\':kwargs[\'quantization_config\']=BitsAndBytesConfig(load_in_8bit=True)\n        return self.transformer_type.from_pretrained(self.folder/\'transformer\',\n            torch_dtype=torch.float16,device_map={\'\':device},low_cpu_mem_usage=True,\n            local_files_only=True,**kwargs)\n\n    def generate(self,prompt,seed=42,steps=None,frames=None,output=None):\n        import torch\n        from video_registry import validate_video\n        from video_precision import move_video_encoder\n        from video_parallel import WanParallelCFG\n        p=self.profile;params=validate_video(prompt,seed,steps,p[\'id\'],frames)\n        if self.pipe is None:self.prepare()\n        for i in range(2):torch.cuda.reset_peak_memory_stats(i)\n        start=time.perf_counter();self.emit(\'encoding\')\n        move_video_encoder(self.encoder,\'cuda:1\')\n        encoded=self.pipe.encode_prompt(prompt=prompt,negative_prompt=\'\',\n            do_classifier_free_guidance=p[\'guidance\']>1,device=torch.device(\'cuda:1\'))\n        if p[\'family\']==\'ltx\':positive,mask,negative,negative_mask=encoded\n        else:positive,negative=encoded[:2]\n        if not torch.isfinite(positive).all() or (negative is not None and not torch.isfinite(negative).all()):\n            raise RuntimeError(\'Non-finite video prompt embeddings\')\n        positive=positive.to(\'cuda:0\');negative=negative.to(\'cuda:0\') if negative is not None else None\n        torch.cuda.synchronize();encoding=time.perf_counter()-start\n        move_video_encoder(self.encoder,\'cpu\');torch.cuda.empty_cache()\n        if self.acceleration==\'split\' and self.replica is not None:\n            self.replica=None;gc.collect();torch.cuda.empty_cache()\n        replica_reused=self.replica is not None;replica_seconds=0;strategy=\'encoder_split\'\n        if p[\'family\']!=\'ltx\' and self.acceleration==\'auto\' and not self.disabled:\n            try:\n                if self.replica is None:\n                    tick=time.perf_counter();self.replica=self.load_transformer(1);replica_seconds=time.perf_counter()-tick\n                strategy=\'cfg_parallel\'\n            except (RuntimeError,ValueError):\n                self.replica=None;self.disabled=True;gc.collect();torch.cuda.empty_cache()\n        latents_finite=[]\n        def progress(pipeline,index,timestep,kwargs):\n            self.emit(\'generating\',step=index+1,steps=params[\'steps\'])\n            if index==params[\'steps\']-1:\n                latents_finite.append(bool(torch.isfinite(kwargs[\'latents\']).all().item()))\n                if not latents_finite[-1]:raise RuntimeError(\'Non-finite video latents\')\n            return kwargs\n        kwargs=dict(prompt_embeds=positive,negative_prompt_embeds=negative,\n            width=p[\'width\'],height=p[\'height\'],num_frames=params[\'frames\'],\n            num_inference_steps=params[\'steps\'],guidance_scale=p[\'guidance\'],\n            generator=torch.Generator(device=\'cuda:0\').manual_seed(seed),\n            callback_on_step_end=progress,output_type=\'np\')\n        if p[\'family\']==\'ltx\':kwargs.update(decode_timestep=.05,decode_noise_scale=.025,\n            prompt_attention_mask=mask.to(\'cuda:0\'),negative_prompt_attention_mask=negative_mask)\n        # Wan2.2 hybrid generation accepts no input image for text-to-video.\n        self.pipe.register_modules(text_encoder=None)\n        sampling=time.perf_counter();fallback=None;cfg_pairs=0\n        try:\n            if strategy==\'cfg_parallel\':\n                try:\n                    router=WanParallelCFG(self.pipe.transformer,self.replica,negative)\n                    try:\n                        with router:result=self.pipe(**kwargs)\n                        cfg_pairs=router.calls\n                    finally:del router\n                except RuntimeError as error:\n                    if not isinstance(error,torch.cuda.OutOfMemoryError):raise\n                    fallback=type(error).__name__;self.replica=None;self.disabled=True\n                if fallback:\n                    gc.collect();torch.cuda.empty_cache();strategy=\'encoder_split\'\n                    kwargs[\'generator\']=torch.Generator(device=\'cuda:0\').manual_seed(seed)\n                    result=self.pipe(**kwargs)\n            else:result=self.pipe(**kwargs)\n        finally:self.pipe.register_modules(text_encoder=self.encoder)\n        torch.cuda.synchronize();sampling_seconds=time.perf_counter()-sampling\n        import numpy as np\n        video=np.asarray(result.frames[0])\n        if not np.isfinite(video).all() or float(video.std())<.001:raise RuntimeError(\'Invalid or blank video\')\n        self.emit(\'exporting\');from diffusers.utils import export_to_video\n        output=Path(output or \'/kaggle/temp/cc-media-video-result.mp4\');output.parent.mkdir(parents=True,exist_ok=True)\n        export_to_video(video,str(output),fps=p[\'fps\'])\n        self.calls+=1\n        return output,dict(model_id=p[\'id\'],media_type=\'video\',width=p[\'width\'],height=p[\'height\'],\n            frames=params[\'frames\'],fps=p[\'fps\'],steps=params[\'steps\'],seed=seed,\n            encoding_seconds=round(encoding,3),generation_seconds=round(sampling_seconds,3),\n            request_seconds=round(time.perf_counter()-start,3),loading_seconds=round(self.load_seconds,3),\n            model_reused=self.calls>1,model_load_count=1,generation_number=self.calls,\n            parallel_strategy=strategy,acceleration_mode=self.acceleration,cfg_pairs=cfg_pairs,\n            replica_load_seconds=round(replica_seconds,3),dual_model_reused=replica_reused,\n            acceleration_fallback=fallback,peak_gpu_gib=[round(torch.cuda.max_memory_allocated(i)/2**30,3) for i in range(2)])\n\n    def close(self):\n        self.pipe=None;self.encoder=None;self.replica=None\n        gc.collect()\n        import torch\n        torch.cuda.empty_cache()\n'}.items():(modules/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(modules))
print('CC Media Studio video environment ready')


In [ ]:
import ipywidgets as widgets
from IPython.display import Video,display,clear_output
from video_registry import VIDEO_MODELS
from video_runtime import VideoRuntime
model=widgets.Dropdown(options=[(p['name'],p['id']) for p in VIDEO_MODELS.values()],value='wan21-t2v-1.3b',description='Model')
prompt=widgets.Textarea(value='An adult anime catgirl reading by a sunlit window, hair moving gently in the breeze. Slow camera movement.',description='Prompt',layout=widgets.Layout(width='95%',height='100px'))
frames=widgets.Dropdown(options=[9,17,33,49,65,81],value=33,description='Frames')
steps=widgets.IntSlider(value=30,min=1,max=50,description='Steps')
seed=widgets.IntText(value=42,description='Seed')
acceleration=widgets.Dropdown(options=[('Auto / 双卡优先','auto'),('Encoder split / 编码分卡','split')],value='auto',description='GPU')
button=widgets.Button(description='Generate / 生成',button_style='primary')
output=widgets.Output();runtime=None
def selected(change):
    p=VIDEO_MODELS[model.value];steps.value=p['default_steps'];steps.disabled=bool(p['fixed_steps'])
model.observe(selected,names='value')
def generate(_):
    global runtime
    button.disabled=True
    try:
        with output:
            clear_output(wait=True)
            if runtime is not None and runtime.profile['id']!=model.value:runtime.close();runtime=None
            if runtime is None:runtime=VideoRuntime(model.value,acceleration=acceleration.value,progress=lambda event:print(event,flush=True)).prepare()
            runtime.acceleration=acceleration.value
            path,usage=runtime.generate(prompt.value,seed=seed.value,steps=steps.value,frames=frames.value,output='/kaggle/working/cc-media-video.mp4')
            display(Video(str(path),embed=True));print(usage)
    finally:button.disabled=False
button.on_click(generate)
display(widgets.VBox([model,prompt,widgets.HBox([frames,steps]),widgets.HBox([seed,acceleration]),button,output]))
